# Evaluierungs-Pipeline

## Alle Evaluierungen zusammengefasst

Was wir haben:
- Bestand: ~140 Charts (Line/Bar/Stacked Bar x simple/complex). 

- Goldstandard (Goldenstandard Subset): Für 27 Charts existieren Golden-Standard-Alt-Texte. 

- Generierte alt-texte (Goldenstandard Subset): Zu den 27 charts haben wir je Temperatur (0.4, default=1.0, 1.6) für das LLM 10 generierte Texte (total generierte alt texte = 27 * 3 * 10). --> pro temperatur 1 generation run id. --> 3 generation_ids nötig für eine vollständige analyse. 

- Menschliche Bewertungen (Subset vom Subset Goldstandard): Für 12 dieser 27 Charts (je 2 pro Kombination aus simple/complex × line/bar/stacked) haben wir 5 Human-Ratings pro Alt-Text zu 4 Kriterien: Klarheit, Kürze, wahrgenommene Vollständigkeit, Neutralität (Skala 1–5 , nur integers).
+ LLM Scores

- LLM-as-a-Judge (Subset vom Subset Goldstandard): Für 27 dieser 27 Charts: 1 Score pro Kriterium und Alt-Text für 6 Kriterien: Klarheit, Kürze, wahrgenommene Vollständigkeit, Neutralität, Vollständigkeit, Faktenkorrektheit. schlussendlich 27(charts) * 3(temp) * 10(alt-text Vrsionen) * 6(verschiedene Kriterien) generierte scores mit begründungen.



Was wir machen sollten:
- RQ1: Howdomanually created gold-standard alt texts compare to LLM-generated alt texts in terms of clarity, conciseness and meaningfulness? 
- RQ2: To what extent do LLM-based evaluations of alt texts align with human expert assessments in terms of clarity, conciseness and meaningfulness? 
- RQ3: How consistent are multiple alt texts generated from the same prompt (inter-generation consistency), and how effectively can embedding-based methods (e.g., SBERT) capture the semantic diversity? 
- RQ4: In what ways does variation in generation parameters affect the quality of LLMgenerated alt texts (with temperature as a case study)? 



Wie wir es machen:
RQ1: 
- Länge (anzahl zeichen) der alt texte (short description metadata, short description overview, long description) pro chart-type und simple/complex (6 Kategorien) (temperature only default)
    - Violinplots oder Boxplots pro Chart-Kategorie (simple/complex × line/bar/stacked)
    - Side-by-side Boxplots: Goldstandard vs. Generated

- LLM scores from LLM-as-a-judge: Vergleiche die scores von den 27 golden standards mit jenen von den 27*10 generierten (temperature only default)
    Für Kriterien: Klarheit, Kürze, wahrgenommene Vollständigkeit, Neutralität, (optional: Vollständigkeit, Faktenkorrektheit)
    - Violinplots oder Boxplots pro Chart-Kategorie (simple/complex × line/bar/stacked)
    - Side-by-side Boxplots: Goldstandard vs. Generated

- SBERT Similarity zwischen den golden standards und den generierten alt texten (27*10)
    Berechne Cosine Similarity zwischen:
    gold_text_i ↔ generated_text_(i,j)
    Für jedes Chart --> 10 Similarity Scores --> Verteilung.
    Plots:
    - Histogram oder Density Plot pro Chart
    - Boxplot „Similarity: Gold vs Generated“ über alle Charts
    - Welche Variante (n_variants) hat den höchsten Similarity Wert? Welches den niedrigsten? Gib die Teste aus.

- UMAP pro chart_id oder alle in einem (27*10 + 27 golden standards)
    Für alle Embeddings: 27 Gold + 27 × 10 Generated
    Ziel: Clustern sich Goldstandards getrennt von LLM-Generierungen?
    Plots
    pro Kategorie (6) eine UMAP:
    - Shape: Gold vs. Generated
    - Farbe: pro chart_id



RQ2: übereinstimmung der ratings von den 5 Menschen verglichen mit den Ratings von 1 LLM as a judge (jede alt text id text wird 1 mal von einem LLM in 6 Kriterien zwischen 0 und 5 bewertet).


RQ3: Konsistenz der alt texte innerhalb der der chart_id (temperatur = default)
- SBERT Similarity zwischen den generierten alt texten (27*10) --> evtl. density/histogram plots für verteilung (dicht = konsistent, breit = inkonsistent)
- UMAP pro chart_id oder alle in einem (27*10)


RQ4: Qualität nach Temperatur:
- LLM-as-judge-scores der 27 charts pro temperatur vergleichen
- alt text längen vergleichen
- alt-text SBERT similarities vergleichen innerhalb der verschiedenen Temperatur gruppen und diese dann wiederum vergleichen


In [1]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import re
import sqlite3
import sys
from matplotlib.ticker import MaxNLocator


In [2]:
current_dir = os.path.dirname(os.path.abspath(__file__)) if '__file__' in globals() else os.getcwd()
src_path = os.path.abspath(os.path.join(current_dir, '..', 'src'))
sys.path.append(src_path)

from e_func_viz_pipeline import *

In [3]:
conn = sqlite3.connect("../chart_database.db")
cursor = conn.cursor()

## DataFrame zusammenstellen

In [4]:
import sqlite3

# 1) Konfiguration
db_path = "../chart_database.db"
generation_run_ids_lst = [1, 2, 4, 6]

# 2) CSV-ähnliches DF aus der DB holen
conn = sqlite3.connect(db_path)
cur = conn.cursor()
csv_df = get_csv_from_db(cur, generation_run_ids_lst)
conn.close()

In [5]:
print(csv_df.head(1).T)

                                                      0
chart_id               006f6723bd41797d2bb1e3f65444757f
alt_text_id                                           1
generation_run_id                                     1
temp                                                1.0
variation_no                                          1
people_evaluation_ids                         1,2,3,4,5
gold_standard_id                                      2
embedding_ids                                        27
metrics_ids                                          25


## RQ1

### Alt-Text Length – Long Description

*(character size; length_tokens_long_boxplot_6cats_hue_source)*

* LLM-generierte Long Descriptions sind durchgehend länger in character size als Goldstandards.
* Der Unterschied ist besonders ausgeprägt bei komplexen Charts (Line, Bar, Stacked Bar).
* Goldstandards zeigen geringere Streuung, was auf einen konsistenteren Stil hindeutet.
* LLM-Texte weisen mehr extreme Ausreisser mit sehr grossen character sizes auf.
* --> Hinweis auf stärker explizite und detaillierte Beschreibungen bei LLMs.



### Alt-Text Length – Metadata

*(character size; length_tokens_metadata_boxplot_6cats_hue_source)*

* Metadata-Texte zeigen sehr ähnliche character sizes für Goldstandards und LLMs.
* Unterschiede sind deutlich kleiner als bei Overview oder Long Description.
* Chart-Komplexität hat nur geringen Einfluss auf die Metadata-Länge.
* Die Varianz ist insgesamt niedrig --> stark strukturierter Texttyp.
* --> Metadata ist der stabilste und konsistenteste Alt-Text-Bestandteil.



### Alt-Text Length – Overview

*(character size; length_tokens_overview_boxplot_6cats_hue_source)*

* LLM-generierte Overviews sind systematisch länger in character size als Goldstandards.
* Goldstandards sind besonders bei einfachen Charts sehr kompakt formuliert.
* LLMs nutzen Overviews häufiger zur zusätzlichen Einordnung und Kontextualisierung.
* Die grössere Streuung bei LLMs weist auf unterschiedliche Abstraktionsstrategien hin.
* --> Overviews tragen wesentlich zu den Längenunterschieden zwischen den Quellen bei.



### Alt-Text Length – Total (Metadata + Overview + Long)

*(character size; length_tokens_total_boxplot_6cats_hue_source)*

* Insgesamt sind LLM-generierte Alt-Texte deutlich länger in character size als Goldstandards.
* Die Differenz nimmt mit zunehmender Chart-Komplexität zu.
* Goldstandards bleiben auch bei komplexen Visualisierungen vergleichsweise kompakt.
* LLM-Texte zeigen eine hohe Varianz, insbesondere bei komplexen Charttypen.
* --> Deutlicher Trade-off zwischen Kürze und Explizitheit.



### LLM-as-Judge Scores – Qualitative Bewertung

*(llm_as_judge_scores_boxplot)*

* Clarity: Goldstandard- und LLM-Texte erreichen vergleichbar hohe Scores.
* Conciseness: Goldstandards schneiden leicht besser ab, der Unterschied ist jedoch moderat.
* Perceived Completeness: LLM-Texte werden häufig als gleichwertig oder vollständiger bewertet.
* Correctness: Keine systematischen Nachteile für LLM-generierte Alt-Texte.
* --> Trotz grösserer character size bleibt die wahrgenommene Qualität hoch.



### Cosine Similarity – Long Description

*(similarity_long_boxplot)*

* Die semantische Ähnlichkeit zwischen LLM- und Goldstandard-Texten ist mittel bis hoch.
* Einfache Charts zeigen höhere Similarity als komplexe.
* Stacked Bar Charts weisen die grösste Varianz auf.
* LLMs verwenden häufig andere sprachliche Strukturen, bei vergleichbarem Inhalt.
* --> Long Descriptions erlauben mehr stilistische Freiheit.



### Cosine Similarity – Metadata

*(similarity_metadata_boxplot)*

* Sehr hohe semantische Ähnlichkeit zwischen LLM- und Goldstandard-Metadata.
* Kaum Unterschiede zwischen Charttypen oder Komplexitätsstufen.
* Sehr geringe Streuung --> hoch standardisierte Inhalte.
* --> LLMs reproduzieren menschliche Metadata nahezu identisch.



### Cosine Similarity – Overview

*(similarity_overview_boxplot)*

* Die Similarity liegt im mittleren Bereich.
* Komplexe Charts zeigen geringere Ähnlichkeit als einfache.
* Unterschiede ergeben sich vor allem durch:

  * Fokuswahl
  * Abstraktionsniveau
  * sprachliche Verdichtung
* --> Overviews sind der interpretativste Alt-Text-Baustein.



### Cosine Similarity – Total Alt-Text

*(similarity_total_boxplot)*

* Die Gesamtsimilarität ist hoch, trotz Unterschiede in einzelnen Komponenten.
* Längere Texte gleichen Abweichungen zwischen Overview und Long Description teilweise aus.
* Unterschiede zwischen Charttypen bleiben sichtbar, sind aber reduziert.
* --> LLM-generierte Alt-Texte sind semantisch nah an Goldstandards, trotz grösserer character size.



### Gesamtaussage (RQ1-relevant)

* **Goldstandards**: kürzer, konsistenter
* **LLM-Texte**: länger in character size, aber klar, korrekt und semantisch nah
* **Ähnlichkeit**: sehr hoch für strukturierte Inhalte, moderat für interpretative Teile


In [6]:
# # 3) RQ1: Daten + Plots
# rq1 = filter_csv_and_get_data_for_rq1_in_db(csv_df, db_path=db_path)

# rq1_lengths_df = rq1.get("lengths_df")
# rq1_scores_df = rq1.get("scores_df")
# rq1_similarity_df = rq1.get("similarity_df")
# rq1_similarity_best_worst = rq1.get("similarity_best_worst")
# rq1_umap = rq1.get("umap")



In [7]:
# visualize_rq1_similarity_boxplot(rq1, outdir="../outputs/eval_figures/rq1")
# visualize_rq1_llm_as_judge(rq1, outdir="../outputs/eval_figures/rq1")
# visualize_rq1_length(rq1, outdir="../outputs/eval_figures/rq1")
# visualize_rq1_length_by_conciseness_hue_source(rq1)

In [8]:
# rq1_scores_df

### Similarity 

Hier sind noch die 12 Alt-Texte von generation run 1 darin. Das heisst pro chart id existieren 10 Text Varianten + z.T noch +1 durch den Generation run 1. Evtl. generation run 1 noch weglassen.

In [9]:
# rq1_similarity_df.sort_values("chart_id")

In [10]:
# rq1_similarity_best_worst.head()

## RQ2: Alignment zwischen Human-Ratings und LLM-as-a-Judge



### Clarity (Detailplot pro Chart)

*(criterion_score_clarity.png)*

* Menschliche Bewertungen liegen überwiegend im oberen Skalenbereich (4–5) über alle Charttypen hinweg.
* LLM-as-a-Judge vergibt ebenfalls überwiegend hohe Scores, zeigt jedoch eine leicht stärkere Differenzierung.
* Bei einzelnen komplexen Charts treten größere Abweichungen zwischen LLM- und Human-Mittelwerten auf.
* Die relative Rangordnung der Charts ist zwischen Humans und LLMs weitgehend konsistent.
* Insgesamt besteht eine hohe Übereinstimmung bei der Bewertung der Klarheit.



### Conciseness (Detailplot pro Chart)

*(criterion_score_conciseness.png)*

* Menschliche Bewertungen bewerten die Alt-Texte nahezu durchgehend als sehr prägnant (meist Score 4–5).
* LLM-as-a-Judge vergibt deutlich niedrigere Scores, häufig im Bereich 1–2.
* Diese Abweichung tritt konsistent über alle Charttypen und Komplexitätsstufen hinweg auf.
* Humans scheinen Verständlichkeit stärker zu berücksichtigen als formale Kürze.
* Es zeigt sich eine systematische Diskrepanz zwischen menschlicher und LLM-basierter Bewertung der Conciseness.



### Neutrality (Detailplot pro Chart)

*(criterion_score_neutrality.png)*

* Menschliche Bewertungen liegen fast durchgehend bei Maximalwerten mit sehr geringer Streuung.
* LLM-as-a-Judge weist eine deutlich größere Varianz auf, insbesondere bei komplexen Charts.
* LLMs bewerten Neutralität offenbar strenger und differenzierter als menschliche Rater.
* Die Abweichungen konzentrieren sich auf spezifische Charts und sind nicht zufällig verteilt.
* Die Übereinstimmung ist moderat, mit einer systematischen Strenge auf Seiten des LLMs.



### Perceived Completeness (Detailplot pro Chart)

*(criterion_score_preceived_completeness.png)*

* Menschliche Bewertungen der wahrgenommenen Vollständigkeit liegen überwiegend im Bereich 4–5.
* LLM-as-a-Judge vergibt ähnliche, jedoch bei komplexen Charts leicht niedrigere Bewertungen.
* Die Abweichungen sind geringer als bei Conciseness, aber stärker ausgeprägt als bei Clarity.
* Die Rangfolge der Charts bleibt größtenteils konsistent zwischen Humans und LLMs.
* Insgesamt zeigt sich eine gute, aber nicht vollständige Übereinstimmung.



### Summary Plot – Humans vs. LLM-as-a-Judge

*(summary.png)*

* Für Clarity besteht eine sehr hohe Übereinstimmung zwischen menschlichen und LLM-basierten Bewertungen.
* Perceived completeness zeigt ebenfalls eine hohe Übereinstimmung mit leichten Abweichungen.
* Neutrality wird von Humans nahezu durchgehend mit Maximalwerten bewertet, während LLMs stärker differenzieren.
* Conciseness weist die größte Diskrepanz auf, mit deutlich niedrigeren Bewertungen durch das LLM.
* Das Alignment zwischen Humans und LLM-as-a-Judge ist stark kriteriumsabhängig.



### Zusammenfassende Interpretation für RQ2

* LLM-as-a-Judge eignet sich gut zur Annäherung an menschliche Bewertungen für Klarheit und wahrgenommene Vollständigkeit.
* Für Conciseness unterscheiden sich die zugrunde liegenden Bewertungsmaßstäbe deutlich zwischen Humans und LLMs.
* LLMs legen stärker formale Kriterien wie Länge und Redundanz zugrunde.
* LLM-as-a-Judge ist kein vollständiger Ersatz für menschliche Bewertungen, aber ein valider Proxy für ausgewählte Qualitätsdimensionen.


In [11]:
# rq2 = filter_csv_and_get_data_for_rq2_in_db(csv_df=csv_df, db_path=db_path)
# visualize_rq2_person_by_criterion(rq2)
# visualize_rq2_summary(rq2)

## RQ3

### RQ3: Pairwise cosine similarities aggregated by chart type × complexity

*(rq3_hist_by_cat6_2x3.png)*

* Die paarweisen Cosine Similarities liegen insgesamt überwiegend im hohen Bereich (ca. 0.8–0.95), was auf eine hohe semantische Konsistenz zwischen mehrfach generierten Alt-Texten hindeutet.
* Einfache Charts zeigen tendenziell höhere Similarities als komplexe Charts, unabhängig vom Charttyp.
* Bar Charts (simple und complex) weisen die höchsten und am stärksten konzentrierten Similarity-Werte auf.
* Line Charts zeigen eine etwas breitere Verteilung, insbesondere bei komplexen Varianten.
* Stacked Bar Charts, vor allem in der komplexen Variante, weisen die größte Streuung auf.
* Die Ergebnisse deuten darauf hin, dass mit zunehmender visueller und semantischer Komplexität die inter-generationale Varianz der Alt-Texte zunimmt.



### RQ3: Pairwise cosine similarities within each chart

*(rq3_hist_per_chart_8x4.png)*

* Innerhalb einzelner Charts sind die Similarity-Verteilungen häufig unimodal und relativ schmal, was auf stabile Generationen aus identischem Prompt hinweist.
* Einige Charts zeigen jedoch deutlich breitere Verteilungen mit niedrigeren Similarity-Werten, insbesondere bei komplexen Line- und Stacked-Bar-Charts.
* Die Varianz zwischen Charts desselben Typs ist teilweise größer als zwischen unterschiedlichen Charttypen.
* Bestimmte Charts weisen konsistent sehr hohe Similarities auf, was auf stark determinierte Beschreibungsstrategien hindeutet.
* Andere Charts zeigen multimodale oder schiefe Verteilungen, was auf alternative inhaltliche Fokussierungen innerhalb der Generationen schließen lässt.
* Insgesamt existiert sowohl chart-abhängige als auch komplexitätsabhängige semantische Diversität zwischen mehrfach generierten Alt-Texten.



### Zusammenfassende Interpretation für RQ3

* Mehrfach generierte Alt-Texte aus identischem Prompt sind überwiegend semantisch konsistent.
* Die verbleibende Variation ist nicht zufällig, sondern systematisch mit Charttyp und Komplexität verknüpft.
* Einfache Charts führen zu stärker konvergenten Alt-Texten, während komplexe Charts mehr Interpretationsspielraum erlauben.
* Embedding-basierte Cosine Similarity erfasst diese Unterschiede zuverlässig und eignet sich zur Quantifizierung inter-generationaler Konsistenz und Diversität.


In [6]:
rq3 = filter_csv_and_get_data_for_rq3_in_db(csv_df=csv_df, db_path=db_path)


In [7]:
visualize_rq3_short_meta(df=rq3, outdir="../outputs/eval_figures/rq3")

visualize_rq3_short_overview(df=rq3, outdir="../outputs/eval_figures/rq3")

visualize_rq3_long_description(df=rq3, outdir="../outputs/eval_figures/rq3")

visualize_rq3_total_text(df=rq3, outdir="../outputs/eval_figures/rq3")



visualize_rq3_umap_meta(df=rq3, outdir="../outputs/eval_figures/rq3")

visualize_rq3_umap_overview(df=rq3, outdir="../outputs/eval_figures/rq3")

visualize_rq3_umap_long(df=rq3, outdir="../outputs/eval_figures/rq3")

visualize_rq3_umap_total(df=rq3, outdir="../outputs/eval_figures/rq3")

ModuleNotFoundError: No module named 'umap'

## RQ4

### RQ4: Length of alt texts across generation temperatures

*(length_vs_temp.png)*

* Die character size der generierten Alt-Texte steigt mit zunehmender Generationstemperatur leicht an.
* Die Medianlängen bei Temperatur 1.0 und 1.6 sind sehr ähnlich, während Temperatur 0.4 etwas kürzere Texte erzeugt.
* Die Streuung der Textlängen nimmt mit höherer Temperatur zu.
* Höhere Temperaturen führen häufiger zu sehr langen Ausreißern.
* Insgesamt hat die Temperatur einen moderaten, aber klar erkennbaren Einfluss auf die Textlänge.



### RQ4: LLM-as-a-judge scores across generation temperatures

*(llm_scores_vs_temp.png)*

* Die Bewertungen für Clarity bleiben über alle Temperaturen hinweg stabil im oberen Skalenbereich.
* Conciseness wird über alle Temperaturen hinweg niedrig bewertet, mit nur minimalen Unterschieden zwischen den Einstellungen.
* Perceived completeness zeigt kaum temperaturabhängige Variation und bleibt konstant hoch.
* Neutrality und Correctness weisen ebenfalls keine systematischen Unterschiede zwischen den Temperaturen auf.
* Insgesamt beeinflusst die Generationstemperatur die durch das LLM-as-a-Judge wahrgenommene Qualität nur geringfügig.



### RQ4: Similarity scores across generation temperatures

*(similarity_scores_vs_temp.png)*

* Die Cosine Similarity zwischen mehrfach generierten Alt-Texten ist bei niedriger Temperatur am höchsten.
* Mit steigender Temperatur sinkt die mediane Similarity leicht, während die Streuung zunimmt.
* Temperatur 0.4 erzeugt die konsistentesten und semantisch ähnlichsten Texte.
* Höhere Temperaturen führen zu mehr semantischer Diversität, jedoch ohne drastische Einbrüche der Similarity.
* Die Ergebnisse zeigen einen klaren Trade-off zwischen Konsistenz und Variation in Abhängigkeit von der Temperatur.


### Zusammenfassende Interpretation für RQ4

* Die Generationstemperatur beeinflusst primär die Länge und semantische Variation der Alt-Texte.
* Qualitätsdimensionen wie Klarheit, Neutralität und wahrgenommene Vollständigkeit bleiben weitgehend stabil.
* Niedrige Temperaturen begünstigen konsistente und kompaktere Texte.
* Höhere Temperaturen erhöhen Diversität und Varianz, ohne die durchschnittliche Qualität deutlich zu verschlechtern.
* Temperatur stellt damit einen effektiven, aber fein zu justierenden Steuerungsparameter für Alt-Text-Generierung dar.

In [23]:
from pathlib import Path

rq4_data = filter_csv_and_get_data_for_rq4_in_db(csv_df, db_path="../chart_database.db")
plot_rq4_lengths_by_temperature(rq4_data)
plot_rq4_similarity_by_temperature(rq4_data)
plot_rq4_llm_scores_by_temperature(rq4_data)
